# Arranque en la nube — hackathon-kit

Corré las celdas en orden. La primera clona el repo y valida que todo funcione.

Si el desafío es de **imágenes**: Entorno de ejecución → Cambiar tipo de entorno → **GPU**.

In [ ]:
REPO = "https://github.com/USUARIO/hackathon-kit.git"   # <-- poner la URL real
RAMA = "herramientas"

import os, subprocess, sys
if not os.path.isdir("hackathon-kit"):
    subprocess.run(["git", "clone", "-b", RAMA, REPO, "hackathon-kit"], check=True)
os.chdir("hackathon-kit")
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install -r requirements.txt
!python bootstrap.py

## Datos del desafío

Los datos **no están en el repo** (`.gitignore`). Subilos a `data/` con el panel
de archivos de Colab, o desde Drive:

```python
from google.colab import drive; drive.mount('/content/drive')
!cp -r "/content/drive/MyDrive/hackathon/data" .
```

## Primera hora: auditoría y detección de trampas

In [ ]:
import pandas as pd, numpy as np
from ic_kit import cleaning, costs, tabular, traps
from ic_kit import submit as sub_mod

TARGET, ID_COL = "nivel_urgencia", "id_paciente"     # <-- ajustar a la consigna
CLASES = costs.LABELS_D1                             # <-- ORDEN DE LA MATRIZ DE COSTOS
COSTO = costs.COST_D1                                # <-- transcribir de la consigna

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
print(tr.shape, te.shape)

aud = cleaning.audit(tr, target=TARGET, df_test=te)
aud[aud.alertas != ""]

In [ ]:
trc, _ = cleaning.auto_clean(tr, target=TARGET)
tec, _ = cleaning.auto_clean(te, verbose=False)
X, y, Xte, ids, clases = tabular.prepare(trc, tec, TARGET, ID_COL, class_order=CLASES)

# VERIFICAR A OJO contra la tabla de la consigna antes de seguir:
print("orden interno de clases:", clases)
pd.DataFrame(COSTO, index=clases, columns=clases).astype(int)

In [ ]:
traps.run_all(X, y, Xte, df_raw=trc, id_col=ID_COL);

## Modelo y decisión sensible al costo

In [ ]:
oof, pte, info = tabular.oof_lgb(X, y, Xte, n_splits=5, seeds=(42, 43, 44))

print(costs.decision_gain(y, oof, COSTO))
print()
print(costs.confusion_cost_report(y, costs.bayes_decision(oof, COSTO), COSTO, labels=clases))

In [ ]:
traps.fit_prior_honest(oof, y, COSTO)
traps.validate_prior_em(oof / oof.sum(1, keepdims=True), y);

## Envío

In [ ]:
pred = costs.bayes_decision(pte, COSTO)
s = sub_mod.make_submission(ids, pred, clases, "work/submit.csv",
                            id_col=ID_COL, target_col=TARGET)
train_dist = pd.Series(np.bincount(y, minlength=len(clases)) / len(y), index=clases)
sub_mod.validate(s, expected_ids=ids, allowed_labels=clases, train_dist=train_dist)

# OJO: lower_is_better segun la direccion de la metrica de ESTE desafio
log = sub_mod.SubmitLog(lower_is_better=False)
print(log.can_submit())

In [ ]:
# despues de subirlo al servidor:
# log.record("work/submit.csv", cv=..., notes="baseline bayes")
# log.set_lb(1, <puntaje del ranking>)
# log.report()

## Cierre: probatorio (30 minutos)

In [ ]:
import numpy as np
np.save("work/oof.npy", oof); np.save("work/y.npy", y)

from ic_kit.probatorio import generar_notebook
generar_notebook(
    grupo="...", integrantes=["Acosta", "Borges", "Pelinski"],
    desafio="...", metrica="... (decir si mayor o menor es mejor)",
    oof="work/oof.npy", y="work/y.npy", C=COSTO, clases=clases,
    trampas=[{"hallazgo": "", "evidencia": "", "decision": ""}],
    hipotesis=[], decisiones=[], descartado=[])